<a href="https://colab.research.google.com/github/Joan2022Laurente/fade/blob/main/notebooks/zImageTurboV2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Git clone the repo and install the requirements. (ignore the pip errors about protobuf)

In [ ]:
# ================================================================
#   SETUP — Dark Beast (Z-Image Turbo) + Qwen3-4B + VAE + LoRA
# ================================================================
import os, subprocess, sys, shutil
from google.colab import userdata

CIVITAI_KEY   = userdata.get('CIVITAI_KEY')
TAILSCALE_KEY = userdata.get('TAILSCALE_KEY')
if not CIVITAI_KEY:   raise RuntimeError("❌ Falta CIVITAI_KEY")
if not TAILSCALE_KEY: raise RuntimeError("❌ Falta TAILSCALE_KEY")
print("✅ Secrets cargados")

COMFY        = "/content/ComfyUI"
MODELS       = f"{COMFY}/models"
CUSTOM_NODES = f"{COMFY}/custom_nodes"
WORKFLOWS    = f"{COMFY}/user/default/workflows"
HF_Z     = "https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files"
HF_TOKEN = userdata.get('HF_TOKEN')
if not HF_TOKEN:
    raise RuntimeError("❌ Falta HF_TOKEN")

def run(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if r.stdout.strip(): print(r.stdout.strip()[-1500:])
    return r.returncode

def pip(pkg):
    run(f'"{sys.executable}" -m pip install {pkg} -q --no-deps 2>/dev/null || '
        f'"{sys.executable}" -m pip install {pkg} -q')

def clone(url, dst):
    if os.path.isdir(dst): run("git pull --ff-only", cwd=dst)
    else: run(f'git clone --depth 1 "{url}" "{dst}"')
    req = f"{dst}/requirements.txt"
    if os.path.exists(req): run(f'"{sys.executable}" -m pip install -r "{req}" -q')

def dl_hf(url, folder, name, token=None):
    os.makedirs(folder, exist_ok=True)
    dest = f"{folder}/{name}"
    if os.path.exists(dest) and os.path.getsize(dest) > 1024*1024:
        print(f"  ✅ {name} ya existe — skip"); return
    print(f"  ↓ {name}")
    auth = f'--header "Authorization: Bearer {token}"' if token else ""
    ret = run(f'aria2c -c -x 8 -s 8 -k 5M --file-allocation=none '
              f'--console-log-level=warn {auth} "{url}" -d "{folder}" -o "{name}"')
    if ret != 0 or not os.path.exists(dest) or os.path.getsize(dest) < 1024*1024:
        wget_auth = f'--header "Authorization: Bearer {token}"' if token else ""
        run(f'wget -q --show-progress {wget_auth} -O "{dest}" "{url}"')
    print(f"  ✅ {name} ({os.path.getsize(dest)/1024**3:.2f}GB)")

def dl_civitai(model_id, params, folder, name):
    os.makedirs(folder, exist_ok=True)
    dest = f"{folder}/{name}"
    if os.path.exists(dest) and os.path.getsize(dest) > 1024*1024:
        print(f"  ✅ {name} ya existe — skip"); return
    print(f"  ↓ {name} (Civitai)...")
    url = f"https://civitai.red/api/download/models/{model_id}?{params}&token={CIVITAI_KEY}"
    run(f'wget -L --progress=dot:giga -O "{dest}" "{url}"')
    sz = os.path.getsize(dest)/1024**3 if os.path.exists(dest) else 0
    print(f"  {'✅' if sz > 1 else '❌'} {name} ({sz:.2f}GB)")

# ── SISTEMA ────────────────────────────────────────────────────
print("\n[SYS] herramientas...")
run("apt-get update -qq && apt-get install -y -qq aria2 git wget curl")

# ── COMFYUI ────────────────────────────────────────────────────
print("\n[0] ComfyUI...")
if not os.path.isdir(COMFY):
    run(f'git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git "{COMFY}"')
else:
    run("git pull --ff-only", cwd=COMFY)
run(f'"{sys.executable}" -m pip install -r "{COMFY}/requirements.txt" -q')
for d in [f"{MODELS}/diffusion_models", f"{MODELS}/text_encoders",
          f"{MODELS}/vae", f"{MODELS}/loras", CUSTOM_NODES, WORKFLOWS]:
    os.makedirs(d, exist_ok=True)

# ── CUSTOM NODES ───────────────────────────────────────────────
print("\n[1] rgthree-comfy...")
clone("https://github.com/rgthree/rgthree-comfy.git", f"{CUSTOM_NODES}/rgthree-comfy")

# ── DEPS ───────────────────────────────────────────────────────
print("\n[2] Deps...")
for p in ["accelerate", "einops", "torchvision", "Pillow"]: pip(p)
print("  ✅")

# ── MODELOS ────────────────────────────────────────────────────
print("\n[3] Modelos...")
dl_hf(f"{HF_Z}/text_encoders/qwen_3_4b.safetensors",
      f"{MODELS}/text_encoders", "qwen_3_4b.safetensors")

dl_hf(f"{HF_Z}/vae/ae.safetensors",
      f"{MODELS}/vae", "ae.safetensors")

dl_civitai("2699886", "type=Model&format=SafeTensor&size=pruned&fp=fp8",
           f"{MODELS}/diffusion_models", "darkbeast_fp8.safetensors")

# ── LORA ───────────────────────────────────────────────────────
print("\n[4] LoRA...")
dl_hf("https://huggingface.co/joanjlau/loraprueba0/resolve/main/YummyHDzit.safetensors",
      f"{MODELS}/loras", "YummyHDzit.safetensors", token=HF_TOKEN)

# ── WORKFLOW ───────────────────────────────────────────────────
src = "/content/workflowDarkBeast.json"
if os.path.exists(src):
    shutil.copy2(src, f"{WORKFLOWS}/workflowDarkBeast.json")
    print("✅ Workflow copiado")
else:
    print("⚠️ Sube workflowDarkBfeast.json a /content/")

# ── VERIFICACIÓN ───────────────────────────────────────────────
print("\n=== VERIFICACIÓN ===")
for path, label in [
    (f"{MODELS}/diffusion_models/darkbeast_fp8.safetensors", "Dark Beast FP8"),
    (f"{MODELS}/text_encoders/qwen_3_4b.safetensors",        "Qwen3-4B encoder"),
    (f"{MODELS}/vae/ae.safetensors",                         "Flux VAE"),
    (f"{MODELS}/loras/YummyHDzit.safetensors",               "LoRA YummyHDzit"),
    (f"{CUSTOM_NODES}/rgthree-comfy",                        "rgthree"),
]:
    ok = os.path.exists(path) and (not os.path.isfile(path) or os.path.getsize(path) > 1024*1024)
    print(f"  {'✅' if ok else '❌'} {label}")

print("\n✅ SETUP COMPLETO")
print("→ KSampler: Steps=8 · CFG=1.0 · euler · beta")

✅ Secrets cargados
  SETUP — Qwen-Image-2512 T2I · GGUF Q4_K_M + Lightning 4s BF16

[SYS] Instalando herramientas del sistema...
  $ apt-get update -qq && apt-get install -y -qq aria2 git wget curl
2_1.36.0-1_amd64.deb ...
Unpacking aria2 (1.36.0-1) ...
Preparing to unpack .../3-libcurl4-openssl-dev_7.81.0-1ubuntu1.24_amd64.deb ...
Unpacking libcurl4-openssl-dev:amd64 (7.81.0-1ubuntu1.24) over (7.81.0-1ubuntu1.23) ...
Preparing to unpack .../4-curl_7.81.0-1ubuntu1.24_amd64.deb ...
Unpacking curl (7.81.0-1ubuntu1.24) over (7.81.0-1ubuntu1.23) ...
Preparing to unpack .../5-libcurl4_7.81.0-1ubuntu1.24_amd64.deb ...
Unpacking libcurl4:amd64 (7.81.0-1ubuntu1.24) over (7.81.0-1ubuntu1.23) ...
Setting up libc-ares2:amd64 (1.18.1-1ubuntu0.22.04.3) ...
Setting up libcurl4:amd64 (7.81.0-1ubuntu1.24) ...
Setting up curl (7.81.0-1ubuntu1.24) ...
Setting up libaria2-0:amd64 (1.36.0-1) ...
Setting up aria2 (1.36.0-1) ...
Setting up libcurl4-openssl-dev:amd64 (7.81.0-1ubuntu1.24) ...
Processing trigg

In [ ]:
# 🔒 Lanzar ComfyUI con Tailscale

import os, time
from google.colab import userdata

TAILSCALE_AUTH_KEY = userdata.get('TAILSCALE_KEY')
if not TAILSCALE_AUTH_KEY:
    raise RuntimeError("❌ Falta TAILSCALE_KEY en Secrets de Colab")

# 1. Instalar Tailscale
!curl -fsSL https://tailscale.com/install.sh | sh

# 2. Iniciar daemon
print("🚀 Iniciando tailscaled...")
!nohup tailscaled --tun=userspace-networking --socks5-server=localhost:1055 > /tmp/tailscaled.log 2>&1 &
time.sleep(5)

# 3. Verificar
print("🔍 Verificando tailscaled...")
!ps aux | grep tailscaled

# 4. Conectar
print("🔗 Conectando a Tailscale...")
!tailscale up --authkey={TAILSCALE_AUTH_KEY} --hostname=colab-comfyui

# 5. Mostrar IP
print("\n" + "="*50)
print("🌐 TU IP PRIVADA DE TAILSCALE ES:")
!tailscale ip -4
print("="*50)

# 6. Lanzar ComfyUI
import torch, gc
gc.collect()
torch.cuda.empty_cache()
print("\n🚀 Iniciando ComfyUI...")
%cd /content/ComfyUI
!python main.py --listen 0.0.0.0 --dont-print-server

Installing Tailscale for ubuntu jammy, using method apt
+ mkdir -p --mode=0755 /usr/share/keyrings
+ curl -fsSL https://pkgs.tailscale.com/stable/ubuntu/jammy.noarmor.gpg
+ tee /usr/share/keyrings/tailscale-archive-keyring.gpg
+ chmod 0644 /usr/share/keyrings/tailscale-archive-keyring.gpg
+ curl -fsSL https://pkgs.tailscale.com/stable/ubuntu/jammy.tailscale-keyring.list
+ tee /etc/apt/sources.list.d/tailscale.list
# Tailscale packages for ubuntu jammy
deb [signed-by=/usr/share/keyrings/tailscale-archive-keyring.gpg] https://pkgs.tailscale.com/stable/ubuntu jammy main
+ chmod 0644 /etc/apt/sources.list.d/tailscale.list
+ apt-get update
Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Get:5 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:6 https://cloud.r-project.org/bin/linux/

In [ ]:

# =========================================================
# COMFYUI + PINGGY TUNNEL
# =========================================================

import subprocess
import threading
import time
import re

# =========================================================
# FUNCIÓN TÚNEL PINGGY
# =========================================================

def run_pinggy():
    print("🌐 Iniciando túnel Pinggy...")

    process = subprocess.Popen(
        [
            "ssh",
            "-p", "443",
            "-o", "StrictHostKeyChecking=no",
            "-R0:localhost:8188",
            "a.pinggy.io"
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    for line in process.stdout:
        print(line.strip())

        match = re.search(r"https://[^\s]+\.a\.free\.pinggy\.link", line)

        if match:
            print("\n" + "=" * 70)
            print("🚀 COMFYUI PUBLIC URL:")
            print(match.group(0))
            print("=" * 70 + "\n")

# =========================================================
# LANZAR TÚNEL
# =========================================================

threading.Thread(
    target=run_pinggy,
    daemon=True
).start()

# =========================================================
# ESPERA
# =========================================================

time.sleep(5)

# =========================================================
# INICIAR COMFYUI
# =========================================================

%cd /content/ComfyUI

!python main.py \
    --listen 0.0.0.0 \
    --port 8188 \
    --dont-print-server

